In [10]:
import pandas as pd
import numpy as np

fy2017 = "/Users/linhchung/Documents/GitHub/Cleaned-OFLC/data/raw/H-1B_Disclosure_Data_FY17.xlsx"

df = pd.read_excel(fy2017)

print(df.shape)
print(df.head())

(624650, 52)
          CASE_NUMBER          CASE_STATUS CASE_SUBMITTED DECISION_DATE  \
0  I-200-16055-173457  CERTIFIED-WITHDRAWN     2016-02-24    2016-10-01   
1  I-200-16064-557834  CERTIFIED-WITHDRAWN     2016-03-04    2016-10-01   
2  I-200-16063-996093  CERTIFIED-WITHDRAWN     2016-03-10    2016-10-01   
3  I-200-16272-196340            WITHDRAWN     2016-09-28    2016-10-01   
4  I-200-15053-636744  CERTIFIED-WITHDRAWN     2015-02-22    2016-10-02   

  VISA_CLASS EMPLOYMENT_START_DATE EMPLOYMENT_END_DATE  \
0       H-1B            2016-08-10          2019-08-10   
1       H-1B            2016-08-16          2019-08-16   
2       H-1B            2016-09-09          2019-09-09   
3       H-1B            2017-01-26          2020-01-25   
4       H-1B            2015-03-01          2018-03-01   

               EMPLOYER_NAME EMPLOYER_BUSINESS_DBA  \
0     DISCOVER PRODUCTS INC.                   NaN   
1           DFS SERVICES LLC                   NaN   
2  EASTBANC TECHNOLOGIES 

In [12]:
print(df.columns.tolist())

['CASE_NUMBER', 'CASE_STATUS', 'CASE_SUBMITTED', 'DECISION_DATE', 'VISA_CLASS', 'EMPLOYMENT_START_DATE', 'EMPLOYMENT_END_DATE', 'EMPLOYER_NAME', 'EMPLOYER_BUSINESS_DBA', 'EMPLOYER_ADDRESS', 'EMPLOYER_CITY', 'EMPLOYER_STATE', 'EMPLOYER_POSTAL_CODE', 'EMPLOYER_COUNTRY', 'EMPLOYER_PROVINCE', 'EMPLOYER_PHONE', 'EMPLOYER_PHONE_EXT', 'AGENT_REPRESENTING_EMPLOYER', 'AGENT_ATTORNEY_NAME', 'AGENT_ATTORNEY_CITY', 'AGENT_ATTORNEY_STATE', 'JOB_TITLE', 'SOC_CODE', 'SOC_NAME', 'NAICS_CODE', 'TOTAL_WORKERS', 'NEW_EMPLOYMENT', 'CONTINUED_EMPLOYMENT', 'CHANGE_PREVIOUS_EMPLOYMENT', 'NEW_CONCURRENT_EMPLOYMENT', 'CHANGE_EMPLOYER', 'AMENDED_PETITION', 'FULL_TIME_POSITION', 'PREVAILING_WAGE', 'PW_UNIT_OF_PAY', 'PW_WAGE_LEVEL', 'PW_SOURCE', 'PW_SOURCE_YEAR', 'PW_SOURCE_OTHER', 'WAGE_RATE_OF_PAY_FROM', 'WAGE_RATE_OF_PAY_TO', 'WAGE_UNIT_OF_PAY', 'H1B_DEPENDENT', 'WILLFUL_VIOLATOR', 'SUPPORT_H1B', 'LABOR_CON_AGREE', 'PUBLIC_DISCLOSURE_LOCATION', 'WORKSITE_CITY', 'WORKSITE_COUNTY', 'WORKSITE_STATE', 'WORKSITE_PO

In [14]:
import pandas as pd

df = pd.read_excel(fy2017)

df["EMPLOYER_NAME"] = (
    df["EMPLOYER_NAME"]
    .astype("string")
    .str.strip()
    .str.upper()
)

keywords = [
    "NVIDIA",
    "APPLE",
    "MICROSOFT",
    "MICRON",
    "ADVANCED MICRO DEVICES",
    "AMAZON",
    "META",
    "FACEBOOK",
    "ALPHABET",
    "GOOGLE",
    "TESLA"
]

mask = df["EMPLOYER_NAME"].str.contains(
    "|".join(keywords),
    na=False,
    regex=True
)

print(
    df.loc[mask, "EMPLOYER_NAME"]
    .value_counts()
    .to_string()
)

EMPLOYER_NAME
GOOGLE INC.                                                      5074
MICROSOFT CORPORATION                                            5005
AMAZON CORPORATE LLC                                             4165
FACEBOOK, INC.                                                   1736
APPLE INC.                                                       1710
TESLA, INC.                                                       816
AMAZON WEB SERVICES, INC.                                         360
MICRON TECHNOLOGY, INC.                                           349
ADVANCED MICRO DEVICES, INC.                                      326
NVIDIA CORPORATION                                                304
GOOGLE INC                                                        188
TESLA MOTORS, INC.                                                151
AMAZON.COM.DEDC, LLC                                              119
MICRONET IT SOLUTIONS, INC.                                       110
AMAZON

In [18]:
company_map = {
    "NVIDIA CORPORATION": "NVIDIA",
    "APPLE INC.": "Apple",
    "MICROSOFT CORPORATION": "Microsoft",
    "MICRON TECHNOLOGY, INC.": "Micron",
    "ADVANCED MICRO DEVICES, INC.": "AMD",
    "AMAZON.COM, INC.": "Amazon",
    "AMAZON.COM SERVICES LLC": "Amazon",
    "META PLATFORMS, INC.": "Meta",
    "FACEBOOK, INC.": "Meta",
    "ALPHABET INC.": "Alphabet",
    "GOOGLE LLC": "Alphabet",
    "TESLA, INC.": "Tesla",
    "TESLA MOTORS, INC.": "Tesla"
}

df["Company"] = df["EMPLOYER_NAME"].map(company_map)

df = df[df["Company"].notna()].copy()

print(df["Company"].value_counts())

Company
Microsoft    5005
Meta         1736
Apple        1710
Tesla         967
Micron        349
AMD           326
NVIDIA        304
Name: count, dtype: int64


In [20]:
df_clean = df.copy()

In [22]:
city_names = {
    "SEATTLE": "Seattle",
    "SAN JOSE": "San Jose",
    "NEW YORK": "New York",
    "AUSTIN": "Austin",
    "DALLAS": "Dallas"
}

# Standardize city names
df["WORKSITE_CITY"] = (
    df["WORKSITE_CITY"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Create the City column
df["City"] = df["WORKSITE_CITY"].map(city_names)

# Keep only the five selected cities
df = df[df["City"].notna()].copy()

print(df["City"].value_counts())

City
Austin      288
Seattle     211
New York    171
Dallas        5
San Jose      3
Name: count, dtype: int64


In [24]:
print(df["WAGE_UNIT_OF_PAY"].value_counts())

WAGE_UNIT_OF_PAY
Year    676
Hour      2
Name: count, dtype: int64


In [26]:
import pandas as pd

# Convert wage values to numbers
df["WAGE_RATE_OF_PAY_FROM"] = pd.to_numeric(
    df["WAGE_RATE_OF_PAY_FROM"],
    errors="coerce"
)

# Standardize wage units
df["WAGE_UNIT_OF_PAY"] = (
    df["WAGE_UNIT_OF_PAY"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Conversion factors
wage_multiplier = {
    "YEAR": 1,
    "MONTH": 12,
    "BI-WEEKLY": 26,
    "WEEK": 52,
    "HOUR": 2080
}

# Create annual wage column
df["Annual_Wage"] = (
    df["WAGE_RATE_OF_PAY_FROM"]
    * df["WAGE_UNIT_OF_PAY"].map(wage_multiplier)
)

# Check results
print(df[[
    "Company",
    "City",
    "WAGE_RATE_OF_PAY_FROM",
    "WAGE_UNIT_OF_PAY",
    "Annual_Wage"
]].head(10))

         Company      City  WAGE_RATE_OF_PAY_FROM WAGE_UNIT_OF_PAY  \
753    Microsoft  New York               135400.0             YEAR   
2239        Meta   Seattle               180000.0             YEAR   
2733       Apple    Austin               120000.0             YEAR   
3453   Microsoft  San Jose               135500.0             YEAR   
4031       Apple    Austin               100693.0             YEAR   
4409         AMD    Austin               102898.0             YEAR   
5000       Apple    Austin               102000.0             YEAR   
7147        Meta  New York               150000.0             YEAR   
11216      Apple    Austin                95000.0             YEAR   
11423  Microsoft    Austin               116680.0             YEAR   

       Annual_Wage  
753       135400.0  
2239      180000.0  
2733      120000.0  
3453      135500.0  
4031      100693.0  
4409      102898.0  
5000      102000.0  
7147      150000.0  
11216      95000.0  
11423     116680.0 

In [19]:
print("Missing wages:", df["Annual_Wage"].isna().sum())

print(df["Annual_Wage"].describe())

Missing wages: 0
count    1.693000e+03
mean     3.144792e+05
std      7.326732e+06
min      5.755400e+04
25%      1.152450e+05
50%      1.350000e+05
75%      1.544000e+05
max      3.016000e+08
Name: Annual_Wage, dtype: float64


In [21]:
df = df.dropna(subset=["Annual_Wage"]).copy()

df = df[df["Annual_Wage"] > 0].copy()

In [30]:
wage_summary = (
    df
    .groupby(["City", "Company"])
    .agg(
        Number_of_Records=("Annual_Wage", "count"),
        Median_Wage=("Annual_Wage", "median"),
        Mean_Wage=("Annual_Wage", "mean")
    )
    .reset_index()
)

print(wage_summary)

        City    Company  Number_of_Records  Median_Wage      Mean_Wage
0     Austin        AMD                115     94640.00   96557.321739
1     Austin      Apple                136     98800.00   97114.610294
2     Austin       Meta                 15     94360.61  100837.129333
3     Austin     Micron                  4    120000.00  117275.000000
4     Austin  Microsoft                  4    116794.00  106022.000000
5     Austin     NVIDIA                 13     94973.00   99837.000000
6     Austin      Tesla                  1    140000.00  140000.000000
7     Dallas      Apple                  3    150000.00  143934.000000
8     Dallas  Microsoft                  2    176066.50  176066.500000
9   New York      Apple                  1    115586.00  115586.000000
10  New York       Meta                127    149158.91  148506.365906
11  New York  Microsoft                 42    140861.00  144367.071429
12  New York     NVIDIA                  1    115398.00  115398.000000
13  Sa

In [32]:
columns_to_keep = [
    "Company",
    "EMPLOYER_NAME",
    "JOB_TITLE",
    "SOC_NAME",
    "City",
    "WAGE_RATE_OF_PAY_FROM",
    "WAGE_UNIT_OF_PAY",
    "Annual_Wage"
]

df_clean = df[columns_to_keep].copy()

df_clean.to_csv(
    "OFLC_BigTech_Wages_2017.csv",
    index=False
)